# Лабораторная работа 18. In-context learning: трансформер как алгоритм обучения (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 6:** интерпретируемость и in-context learning

**Цель.** Проверить экспериментально, что трансформер, обученный на последовательностях задач, выполняет «обучение в контексте» как алгоритм, сравнимый с классическими методами. Также изучить, как предобученные LLM используют метки демонстраций.

**Что нужно сдать:**
1. Трансформер для in-context линейной регрессии (Garg et al., 2022) и сравнение с OLS, ridge, kNN и одним шагом градиентного спуска.
2. Однослойное линейное attention и его сравнение с конструкцией «один шаг GD» (von Oswald et al., 2023).
3. Поведение при сдвигах распределения.
4. Few-shot с верными, перевёрнутыми и бессмысленными метками на предобученной LLM (Min et al., 2022; Wei et al., 2023).

**Теория.** Задача: \(w \sim \mathcal{N}(0, I_d)\), \(x_i \sim \mathcal{N}(0, I_d)\), \(y_i = w^\top x_i\). Последовательность \((x_1, y_1, \dots, x_k, y_k, x_q)\), модель предсказывает \(y_q\). Байесовски оптимальный ответ без шума при \(k \ge d\) — OLS. Линейное self-attention с токенами \(z_i = (x_i, y_i)\) вычисляет
\[
\hat y_q = e_y^\top W_{PV}\Big(\frac1k\sum_i z_i z_i^\top\Big) W_{KQ}\, z_q ,
\]
и при \(W_{KQ} = \begin{pmatrix} I & 0\\ 0 & 0\end{pmatrix}\), \(W_{PV} = \eta\, e_y e_y^\top\) это ровно один шаг GD из \(w_0 = 0\): \(\hat y_q = \eta\, x_q^\top\frac1k\sum_i y_i x_i\).

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

In [ ]:
# ==== Эталонный mini-GPT (решение ЛР 6), используется как готовый модуль ====
from dataclasses import dataclass, asdict

class CharTokenizer:
    def __init__(self, text):
        self.chars = sorted(set(text)); self.stoi = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
    def encode(self, s): return [self.stoi[c] for c in s if c in self.stoi]
    def decode(self, ids): return "".join(self.chars[i] for i in ids)

@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.0

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def rope_cache(T, d_head, base=10000.0, device="cpu", scale=1.0):
    inv = 1.0 / (base ** (torch.arange(0, d_head, 2, device=device).float() / d_head))
    t = torch.arange(T, device=device).float() / scale
    ang = torch.outer(t, inv)                        # (T, d_head/2)
    return ang.cos(), ang.sin()

def apply_rope(x, cos, sin):                         # x: (B, H, T, d_head)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head, self.d_head = cfg.n_head, cfg.n_embd // cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.d_head).transpose(1, 2) for t in (q, k, v))
        q, k = apply_rope(q, cos[:T], sin[:T]), apply_rope(k, cos[:T], sin[:T])
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.w1, self.w3 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False)
    def forward(self, x): return self.w2(F.silu(self.w1(x)) * self.w3(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1, self.attn = RMSNorm(cfg.n_embd), CausalSelfAttention(cfg)
        self.n2, self.mlp = RMSNorm(cfg.n_embd), SwiGLU(cfg.n_embd, int(8 * cfg.n_embd / 3))
    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)
        return x + self.mlp(self.n2(x))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.norm = RMSNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.head.weight = self.wte.weight                    # weight tying
        cos, sin = rope_cache(4 * cfg.block_size, cfg.n_embd // cfg.n_head)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
        self.apply(self._init)
        for n, p in self.named_parameters():                  # масштабирование выходных проекций по глубине
            if n.endswith("proj.weight") or n.endswith("w2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.n_layer))
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)): nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        x = self.wte(idx)
        for b in self.blocks: x = b(x, self.cos, self.sin)
        logits = self.head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1] / temperature, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def get_batch(data, block_size, batch_size, device=DEVICE):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, data, block_size, batch_size=32, iters=20):
    model.eval(); losses = [model(*get_batch(data, block_size, batch_size))[1].item() for _ in range(iters)]
    model.train(); return float(np.mean(losses))

def train_lm(model, train_data, val_data, steps, lr=3e-3, batch_size=32, warmup=50, log_every=100):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
    hist = []
    for s in range(steps):
        x, y = get_batch(train_data, model.cfg.block_size, batch_size)
        _, loss = model(x, y); opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if s % log_every == 0 or s == steps - 1:
            hist.append((s, loss.item(), estimate_loss(model, val_data, model.cfg.block_size)))
    return hist

def get_reference_gpt(text, cfg=None, steps=None, name=None):
    """Загружает или обучает эталонную символьную модель на тексте."""
    tok = CharTokenizer(text)
    data = torch.tensor(tok.encode(text), dtype=torch.long)
    n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
    cfg = cfg or (GPTConfig(tok.vocab_size, 64, 2, 4, 64) if FAST_DEV_RUN else GPTConfig(tok.vocab_size, 256, 6, 6, 384))
    name = name or f"ref_gpt_L{cfg.n_layer}_d{cfg.n_embd}_T{cfg.block_size}.pt"
    path = ARTIFACTS / name
    model = GPT(cfg).to(DEVICE)
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=DEVICE))
    else:
        train_lm(model, train_data, val_data, steps or (300 if FAST_DEV_RUN else 5000))
        torch.save(model.state_dict(), path)
    return model, tok, train_data, val_data

In [ ]:
# ==== Загрузка предобученных моделей Hugging Face ====
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_lm(name, dtype=None):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    dtype = dtype or (torch.float16 if DEVICE == "cuda" else torch.float32)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype)
    return model.to(DEVICE).eval(), tok

@torch.no_grad()
def perplexity(model, tok, text, max_len=512, stride=256, max_windows=None):
    ids = tok(text, return_tensors="pt").input_ids.to(DEVICE)
    nll, cnt, prev = 0.0, 0, 0
    for i, start in enumerate(range(0, ids.size(1) - 1, stride)):
        if max_windows and i >= max_windows: break
        end = min(start + max_len, ids.size(1)); x = ids[:, start:end]
        tgt = x.clone(); tgt[:, : max(0, prev - start)] = -100
        out = model(x, labels=tgt)
        n = (tgt[:, 1:] != -100).sum().item(); nll += out.loss.float().item() * n; cnt += n; prev = end
        if end == ids.size(1): break
    return math.exp(nll / cnt)

## 1. Данные и трансформер для in-context регрессии

Токены: \(x_i\) кодируется как \((x_i, 0)\), \(y_i\) — как \((0, \dots, 0, y_i)\), размерность \(d+1\). Модель — каузальный трансформер из ЛР 6 (блоки `Block` с RoPE) с линейными слоями чтения и записи. Loss — среднее MSE предсказаний \(y_i\) на позициях всех \(x_i\) (то есть для каждого префикса из \(0..k-1\) примеров).

In [ ]:
D = 5 if FAST_DEV_RUN else 10
K = 2 * D + 5                                      # число примеров в контексте при обучении

def sample_tasks(B, k, d=D, x_scale=1.0, noise=0.0, gen=None):
    w = torch.randn(B, d, 1, generator=gen); x = torch.randn(B, k, d, generator=gen) * x_scale
    y = (x @ w).squeeze(-1) + noise * torch.randn(B, k, generator=gen)
    return x, y, w

def to_tokens(x, y):
    B, k, d = x.shape
    tx = torch.cat([x, torch.zeros(B, k, 1)], -1); ty = torch.cat([torch.zeros(B, k, d), y[..., None]], -1)
    return torch.stack([tx, ty], 2).reshape(B, 2 * k, d + 1)              # x1, y1, x2, y2, ...

class ICLTransformer(nn.Module):
    def __init__(self, d, n_embd=128, n_layer=4, n_head=4, max_len=256):
        super().__init__()
        cfg = GPTConfig(vocab_size=1, block_size=max_len, n_layer=n_layer, n_head=n_head, n_embd=n_embd)
        self.read_in = nn.Linear(d + 1, n_embd); self.blocks = nn.ModuleList(Block(cfg) for _ in range(n_layer))
        self.norm = RMSNorm(n_embd); self.read_out = nn.Linear(n_embd, 1)
        cos, sin = rope_cache(max_len, n_embd // n_head); self.register_buffer("cos", cos); self.register_buffer("sin", sin)
    def forward(self, x, y):
        # TODO: ваш код
        raise NotImplementedError

set_seed(0); icl = ICLTransformer(D).to(DEVICE); opt = torch.optim.AdamW(icl.parameters(), lr=1e-3, weight_decay=0.0)
STEPS = 2000 if FAST_DEV_RUN else 20000
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=STEPS, pct_start=0.05)
t0 = time.time(); hist = []
for s in range(STEPS):
    # TODO: ваш код
    raise NotImplementedError
    hist.append(loss.item())
    if (s + 1) % 500 == 0: print(f"шаг {s + 1}: MSE {np.mean(hist[-100:]):.3f}  ({time.time() - t0:.0f} c)")
_ = icl.eval()

## 2. Сравнение с классическими алгоритмами

Для числа примеров \(i = 0..K-1\) посчитайте среднюю нормированную ошибку \((\hat y - y)^2/d\) предсказания \(y_{i+1}\) по первым \(i\) примерам для:
- трансформера;
- OLS (решение минимальной нормы через `torch.linalg.lstsq` / псевдообратную);
- ridge с \(\lambda = 0.1\);
- kNN (\(k = 3\), среднее \(y\) ближайших соседей);
- одного шага GD из нуля с оптимальным \(\eta\), подобранным на обучающих задачах для каждого \(i\);
- нулевого предсказания (базовая линия, ошибка равна 1).

In [ ]:
def baselines(x, y, i, method, lam=0.1, eta=None):
    """Предсказание y[:, i] по (x[:, :i], y[:, :i])."""
    xq = x[:, i]
    if i == 0: return torch.zeros(len(x))
    X, Y = x[:, :i], y[:, :i]
    # TODO: ваш код
    raise NotImplementedError

def fit_eta(i, n=4096):
    x, y, _ = sample_tasks(n, i + 1); g = (y[:, :i, None] * x[:, :i]).mean(1); a = (x[:, i] * g).sum(-1)
    return ((a * y[:, i]).sum() / (a * a).sum()).item()          # МНК по η

ETAS = {i: fit_eta(i) for i in range(1, K)}
print("оптимальные η:", {i: round(e, 2) for i, e in list(ETAS.items())[:6]}, "... теория k/(k+d+1):", [round(i / (i + D + 1), 2) for i in range(1, 7)])

@torch.no_grad()
def error_curves(x, y, model=icl, k=K):
    curves = {}
    curves["трансформер"] = ((model(x, y).cpu() - y) ** 2).mean(0)[:k].numpy() / D
    for m in ["OLS", "ridge", "kNN", "GD"]:
        curves[m] = np.array([((baselines(x, y, i, m, eta=ETAS.get(i, 1.0)) - y[:, i]) ** 2).mean().item() / D for i in range(k)])
    return curves

gen = torch.Generator().manual_seed(123)
xt, yt, _ = sample_tasks(2000, K, gen=gen)
curves = error_curves(xt, yt)
plt.figure(figsize=(7, 4))
for name, c in curves.items(): plt.plot(c, label=name)
plt.axhline(1.0, color="gray", ls=":", label="нулевое предсказание"); plt.axvline(D, color="k", ls="--", lw=0.8)
plt.ylim(-0.05, 1.5); plt.xlabel("число примеров в контексте"); plt.ylabel("ошибка / d"); plt.legend(); plt.show()

## 3. Однослойное линейное attention против одного шага GD

Обучите модель с одним слоем линейного attention (без softmax, без MLP и нормировок), параметры \(W_{KQ}, W_{PV} \in \mathbb{R}^{(d+1)\times(d+1)}\), на задачах с фиксированным \(k = K\) примерами (предсказание только для запроса). Затем:
1. сравните её ошибку с одним шагом GD с оптимальным \(\eta\);
2. посчитайте MSE между предсказаниями LSA и GD на новых задачах;
3. визуализируйте матрицы и объясните структуру, предсказанную теорией (\(W_{KQ}\) — блок \(\propto I\), у \(W_{PV}\) значим только элемент для \(y\)).

In [ ]:
class LinearSelfAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W_kq = nn.Parameter(torch.randn(d + 1, d + 1) * 0.01); self.W_pv = nn.Parameter(torch.randn(d + 1, d + 1) * 0.01)
    def forward(self, x, y):
        # TODO: ваш код
        raise NotImplementedError

set_seed(0); lsa = LinearSelfAttention(D); opt = torch.optim.Adam(lsa.parameters(), lr=1e-2)
for s in range(3000 if FAST_DEV_RUN else 10000):
    x, y, _ = sample_tasks(256, K + 1); loss = F.mse_loss(lsa(x, y), y[:, -1])
    opt.zero_grad(); loss.backward(); opt.step()

# TODO: ваш код (ошибка LSA и GD, MSE между ними, визуализация матриц)

## 4. Сдвиги распределения

Сравните ошибку трансформера и OLS при:
1. масштабе входов \(x \cdot 2\);
2. шуме в ответах \(\sigma = 0.5\) (сравните также с ridge);
3. длине контекста \(2K\) (больше, чем при обучении).

Сделайте вывод: какой алгоритм «выучил» трансформер и где это сходство ломается.

In [ ]:
# TODO: ваш код (три сдвига, графики ошибок трансформера, OLS и ridge)

## 5. Метки в демонстрациях у предобученной LLM

Задача — тональность коротких отзывов. Для \(k \in \{0, 4, 8, 16\}\) демонстраций (в `FAST_DEV_RUN` — \(\{0, 4, 8\}\), 2 набора демонстраций) сравните точность при:
- верных метках («positive»/«negative»);
- перевёрнутых метках (все метки демонстраций инвертированы; оцениваем, следует ли модель инвертированному отображению — точность относительно перевёрнутой разметки);
- семантически не связанных метках («foo»/«bar»).

Предсказание — сравнение логитов первых токенов двух вариантов метки после «Sentiment:» (один прямой проход на пример). Усредните по 5 случайным наборам демонстраций. Сравните две модели разного размера.

In [ ]:
POS = ["I loved this movie, it was wonderful.", "A brilliant and touching story.", "The acting was superb and the plot gripping.",
       "One of the best films I have seen this year.", "Absolutely delightful from start to finish.", "A beautiful, heartwarming experience.",
       "The soundtrack is fantastic and the visuals are stunning.", "I would happily watch it again.", "Smart, funny and full of charm.",
       "An inspiring film with great performances.", "The director did an amazing job.", "It exceeded all my expectations.",
       "A masterpiece of modern cinema.", "Great characters and a satisfying ending.", "I laughed and cried, truly excellent.",
       "Fresh, original and very entertaining.", "The cast is wonderful and the script is sharp.", "A joy to watch.",
       "Highly recommended for the whole family.", "Moving, powerful and unforgettable.", "Everything about it works perfectly.",
       "A clever and enjoyable ride.", "The best performance of his career.", "Pure fun, I enjoyed every minute."]
NEG = ["I hated this movie, it was terrible.", "A boring and predictable story.", "The acting was awful and the plot made no sense.",
       "One of the worst films I have seen this year.", "Painfully dull from start to finish.", "A depressing, empty experience.",
       "The soundtrack is annoying and the visuals are cheap.", "I would never watch it again.", "Stupid, unfunny and lifeless.",
       "A disappointing film with weak performances.", "The director did a terrible job.", "It failed all my expectations.",
       "A complete waste of time.", "Flat characters and a frustrating ending.", "I was bored and fell asleep.",
       "Stale, derivative and very tedious.", "The cast is wasted and the script is lazy.", "A chore to watch.",
       "Not recommended for anyone.", "Forgettable, weak and pointless.", "Nothing about it works at all.",
       "A clumsy and tiresome mess.", "The worst performance of his career.", "Pure torture, I regretted every minute."]
data = [(s, 1) for s in POS] + [(s, 0) for s in NEG]
rng = random.Random(0); rng.shuffle(data); demo_pool, test_items = data[:28], data[28:]

LLM_NAMES = ["HuggingFaceTB/SmolLM2-135M", "HuggingFaceTB/SmolLM2-360M"] if FAST_DEV_RUN else ["Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-7B"]
LABELS = {"верные": ("negative", "positive"), "перевёрнутые": ("positive", "negative"), "foo/bar": ("bar", "foo")}

@torch.no_grad()
def icl_accuracy(model, tok, k, label_words, seed):
    # TODO: ваш код (промпт из k демонстраций «Review: ...\nSentiment: <метка>\n\n», затем тестовый отзыв; один прямой проход, сравните логиты первых токенов двух меток)
    raise NotImplementedError

import pandas as pd
rows = []
for name in LLM_NAMES:
    lm, tk = load_lm(name)
    t0 = time.time()
    for setting, words in LABELS.items():
        for k in ((0, 4, 8) if FAST_DEV_RUN else (0, 4, 8, 16)):
            if k == 0 and setting != "верные": continue
            accs = [icl_accuracy(lm, tk, k, words, seed) for seed in range(1 if k == 0 else (2 if FAST_DEV_RUN else 5))]
            rows.append({"модель": name.split("/")[-1], "метки": setting, "k": k, "точность": np.mean(accs)})
    print(f"{name}: {time.time() - t0:.0f} c"); del lm
icl_df = pd.DataFrame(rows); display(icl_df.pivot_table(index=["модель", "метки"], columns="k", values="точность").round(2))

## Вопросы

1. Почему ошибка трансформера резко падает около \(k = d\)? Что делает OLS при \(k < d\) и почему трансформер там ведёт себя как ridge, а не как OLS?
2. Докажите, что линейное self-attention с указанными \(W_{KQ}, W_{PV}\) реализует один шаг GD. Почему многослойная модель может реализовать несколько шагов (или предобусловленный GD)?
3. Почему трансформер хуже обобщается на \(x\cdot 2\), чем OLS? Что это говорит о том, какой алгоритм он реализует?
4. Что показывает эксперимент с перевёрнутыми метками: использует ли модель отображение «вход → метка» из демонстраций или лишь формат и пространство меток? Как это зависит от размера модели?
5. Какие выводы о «механизме» in-context learning в больших LLM можно и нельзя сделать из экспериментов с игрушечной регрессией?